# 08 - Discrete-time hazard model (survival framing)

The static classifiers (01-03) score `P(cancel before arrival)` from features known at booking time. This means that the same booking gets the same score regardless of how
close arrival is. In reality, the cancel risk is highly dependent on the days left until booking. This notebook builds a discrete-time **hazard** that re-scores every day.

`h(d) = P(cancel in the window ending d days before arrival | still open at d)`

A booking contributes one row per snapshot it survives to. The snapshot grid is **daily for d = 1..14** (the decision horizon) plus a **coarse tail** (21..270) so long-lead cancellations are not mislabelled as survivors. Per-booking `P(cancel before arrival)` is the survival product `1 - prod(1 - h(d))` over the booking's remaining snapshots.

The heavy logic lives in `src/hazard.py` (unit-tested). This notebook drives it and reports metrics. No-shows are **censored survivors** (reached arrival without cancelling).

## 0 - Setup

In [10]:
from __future__ import annotations
import sys, time
from pathlib import Path
_t0 = time.perf_counter()
def _step(m): print(f"  [{time.perf_counter()-_t0:5.2f}s] {m}", flush=True)

_step("locating project root...")
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists():
    if _here == _here.parent:
        raise RuntimeError("could not find project root")
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from src import (load_clean_reservations, color, data_dir, figures_dir, tables_dir)
from src import walkforward as WF
import src.hazard as HZ

pio.templates.default = "plotly_white"
BRAND = {n: color(n) for n in ["yellow", "blue", "green", "orange", "pink", "purple", "red"]}
FIG_DIR = figures_dir() / "08_hazard"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = tables_dir()  / "08_hazard"; TBL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 200)
_step("setup done (plotly).")

  [ 0.00s] locating project root...
  [ 0.56s] setup done (plotly).


## 1 - Load cleaned data + build the survival event columns

`src.hazard.add_event_columns` reads the canonical `cancel_days_before_arrival` from the clean parquet (built in 00 §3.2).

In [11]:
clean = load_clean_reservations()
ce = HZ.add_event_columns(clean)

n_book = len(ce)
n_evt  = int(ce["event_d"].notna().sum())
print(f"resolved bookings:                 {n_book:,}")
print(f"pre-arrival cancel events:         {n_evt:,}  ({n_evt / n_book:.1%})")
print(f"censored survivors (incl no-show): {n_book - n_evt:,}")

ev = ce.loc[ce["event_d"].notna(), "event_d"]
print(f"event timing (days before arrival): median={ev.median():.0f}  "
      f"p90={ev.quantile(0.9):.0f}  max={ev.max():.0f}")

fig = go.Figure()
fig.add_histogram(x=ev.clip(upper=180), nbinsx=60, marker_color=BRAND["blue"])
fig.update_layout(title="When do cancellations happen? (days before arrival)",
                  xaxis_title="days before arrival (capped at 180)",
                  yaxis_title="cancellations", bargap=0.02)
fig.show()

resolved bookings:                 169,151
pre-arrival cancel events:         35,118  (20.8%)
censored survivors (incl no-show): 134,033
event timing (days before arrival): median=7  p90=56  max=364


## 2 - Person-period grid + empirical hazard by horizon

Each booking is expanded to one row per snapshot it survives to where the target is "cancelled within this snapshot's window". Because the coarse snapshots span wider windows, I plot the **per-day** hazard (window rate / window width) so daily and coarse snapshots are comparable.

In [12]:
num, cat = HZ.feature_lists(clean)
print(f"features: {len(num)} numeric + {len(cat)} categorical")
print(f"snapshot grid (daily 1..14 + coarse tail): {HZ.SNAP}")

pp, _ = HZ.build_person_period(ce, num, cat)
g = pp.groupby(HZ.AXIS).agg(haz=("y", "mean"), width=("width", "first"), n=("y", "size"))
g["haz_per_day"] = g["haz"] / g["width"]          # width-normalised => comparable across snapshots
print(f"person-period rows: {len(pp):,}")

fig = go.Figure()
fig.add_bar(x=g.index, y=g["haz_per_day"], marker_color=BRAND["blue"])
fig.update_layout(title="Per-day cancellation hazard by days-until-arrival (width-normalised)",
                  xaxis_title="days until arrival (snapshot d)",
                  yaxis_title="P(cancel per day | at risk)")
fig.update_xaxes(autorange="reversed")
fig.show()

features: 18 numeric + 7 categorical
snapshot grid (daily 1..14 + coarse tail): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 21, 30, 45, 60, 90, 120, 180, 270]
person-period rows: 1,391,411


## 3 - Fit + persist the deployment hazard model

`retrain_hazard` fits on **all resolved** data with a RandomizedSearch (XGBoost, early stopping) and **per-snapshot-band isotonic calibration** (a daily-band map and a coarse-tail map - a wide-window hazard is not on the same scale as a dailyone, so one pooled map would miscalibrate both), then persists the joblib + card that the app's retrain path uses. We reload the artifact for the diagnostics below.

In [13]:
_step("fitting + persisting deployment hazard (early stopping; minutes)...")
res = HZ.retrain_hazard(seed=RANDOM_STATE)
hz  = HZ.load_hazard()
print("persisted ->", res["persisted"]["joblib"])
print("card      ->", res["persisted"]["card"])
print(f"val AP = {hz['val_ap']:.4f} | best_iteration = {hz['best_iteration']} | "
      f"person-period train rows = {hz['n_train_pp']:,}")
print(f"chosen hp: {hz['hp']}")
print(f"calibration bands: edge d={hz['iso_bands']['edge']} "
      f"(separate daily / coarse-tail maps)")

  [ 4.33s] fitting + persisting deployment hazard (early stopping; minutes)...
persisted -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/Data/08_hazard_model.joblib
card      -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/reports/tables/08_hazard/model_card.json
val AP = 0.0728 | best_iteration = 242 | person-period train rows = 1,210,650
chosen hp: {'max_depth': 7, 'learning_rate': 0.04184263761235187, 'min_child_weight': 20, 'reg_lambda': 2.8171515121115074, 'subsample': 0.945807721451697, 'colsample_bytree': 0.8166907233119949}
calibration bands: edge d=14 (separate daily / coarse-tail maps)


## 4 - Per-band calibration maps

The two isotonic maps translate the raw XGBoost score into a calibrated hazard. They differ because the daily and coarse-tail snapshots carry different window widths and base rates - the fix for the old single-pooled-map miscalibration.

In [14]:
b = hz["iso_bands"]
grid = np.linspace(0.0, 0.35, 200)
fig = go.Figure()
fig.add_scatter(x=grid, y=b["le"].predict(grid), mode="lines",
                line=dict(color=BRAND["blue"]),   name=f"daily band (d <= {b['edge']})")
fig.add_scatter(x=grid, y=b["gt"].predict(grid), mode="lines",
                line=dict(color=BRAND["orange"]), name=f"coarse tail (d > {b['edge']})")
fig.add_scatter(x=grid, y=grid, mode="lines",
                line=dict(color="grey", dash="dash"), name="identity")
fig.update_layout(title="Per-band isotonic calibration (raw score -> calibrated hazard)",
                  xaxis_title="raw model score", yaxis_title="calibrated hazard")
fig.show()

## 5 - Matched walk-forward: hazard vs static (apples-to-apples)

Both models are scored on the **same** arrival-anchored rows (active at S, arriving within 14 days) against the **same** label (`cancel before arrival`), so they estimate the same quantity. `step_days = horizon` tiles the timeline contiguously (as in 00 §12). The promotion signal is `mean(ΔAUC) > its own std` (beats noise), not a magic 0.01 gate.

In [15]:
_step("matched arrival-anchored walk-forward (hazard vs static; heavy)...")
wf = HZ.walk_forward_eval_hazard(n_folds=6, horizon_days=14, step_days=14,
                                 compare_static=True, seed=RANDOM_STATE)
pf = pd.DataFrame(wf["per_fold"])
display(pf)
print("aggregate:", {k: round(v["mean"], 4) for k, v in wf["aggregate"].items()})
print("promotion gate (mean ΔAUC > its std):", wf["gate"])
pf.to_csv(TBL_DIR / "hazard_vs_static_walkforward.csv", index=False)

if "auc_static" in pf.columns:
    fig = go.Figure()
    fig.add_scatter(x=pf["S"], y=pf["auc_haz"],    mode="lines+markers",
                    line=dict(color=BRAND["blue"]),   name="hazard")
    fig.add_scatter(x=pf["S"], y=pf["auc_static"], mode="lines+markers",
                    line=dict(color=BRAND["orange"]), name="static baseline")
    fig.update_layout(title="Matched walk-forward AUC (same rows / same label)",
                      xaxis_title="scoring date S", yaxis_title="AUC (cancel before arrival)")
    fig.show()

  [465.43s] matched arrival-anchored walk-forward (hazard vs static; heavy)...


NameError: name '_load_clean' is not defined

## 6 - Per-arrival-night calibration (leak-free)

Aggregate per-booking cancel probabilities to **expected freed rooms per arrival night**, with the Poisson-binomial variance `sum p(1-p)`. To stay leak-free we refit the hazard on the last fold's *train* (resolved by S) and score that fold's *test* nights. `coverage_report` also returns the overdispersion factor phi (phi>1 => correlated cancellations => widen intervals by sqrt(phi)).

In [ ]:
cl = WF.add_outcome_known_date(clean)
folds = WF.make_folds(cl, n_folds=6, horizon_days=14, step_days=14)
f = folds[-1]
tr, te = cl.iloc[f.train_idx].copy(), cl.iloc[f.test_idx].copy()
S = pd.Timestamp(f.origin)

_step("refit on fold train (leak-free per-night calibration)...")
hz_S = HZ.fit_hazard(tr, seed=RANDOM_STATE)

arr = pd.to_datetime(te["arrival"], utc=True); cre = pd.to_datetime(te["created"], utc=True)
te["lead"]  = (arr - cre) / pd.Timedelta(days=1)
te[HZ.AXIS] = ((arr - S) / pd.Timedelta(days=1)).clip(lower=1)          # remaining days to arrival
p = HZ.survival_cancel_proba(te, HZ.hazard_fn(hz_S), hz_S["num"], hz_S["cat"],
                             hz_S["cat_dtypes"], snaps=hz_S.get("snap"))
y = (pd.to_numeric(te["status"], errors="coerce").fillna(0).astype(int).to_numpy() == 1)

pn = HZ.per_night_table(te, p, label=y.astype(int))
cov = HZ.coverage_report(pn)
print("per-night coverage / overdispersion:",
      {k: (round(v, 3) if isinstance(v, float) else v) for k, v in cov.items()})
print(f"aggregate recalibration factor (val-style): "
      f"{HZ.recalibration_factor(pn):.3f}")

pn2 = pn.sort_values("arrival_date")
fig = go.Figure()
fig.add_scatter(x=pn2["arrival_date"], y=pn2["exp"], mode="lines",
                line=dict(color=BRAND["blue"]), name="expected freed")
fig.add_scatter(x=pn2["arrival_date"], y=pn2["act"], mode="markers",
                marker=dict(color=BRAND["orange"], size=5), name="actual freed")
fig.update_layout(title="Per-arrival-night expected vs actual freed rooms (held-out fold)",
                  xaxis_title="arrival night", yaxis_title="rooms freed (cancel before arrival)")
fig.show()

## 7 - Honest review

**What the hazard model gives us that the static one can't.** Risk *now*,
conditioned on still being open d days out; the correct rising-risk curve toward
arrival (§2); a natural slot for time-varying features (calendar / market /
property events); and per-night expected-freed rooms with a proper
Poisson-binomial uncertainty for the overbooking allowance (§6).

**Design decisions (resolved).**
- Snapshot grid is daily 1..14 (decision horizon) + coarse tail to 270, so
  long-lead cancellations are events, not mislabelled survivors.
- XGBoost with early stopping (no fixed tree count); small RandomizedSearch.
- Calibration is **per snapshot band** (daily vs coarse tail) - one pooled
  isotonic over heterogeneous window widths miscalibrated both.
- Comparison to the static baseline is on the **same rows and label** (§5), so
  it is a like-for-like estimand, not the old P(cancel-ever)-vs-per-window mix.
- No-shows are censored survivors, coherent with 00's target.

**Open / next.**
- Target encoding of `property_name` (and other high-card categoricals) may beat
  OHE for the hazard model - benchmark before adopting.
- Time-varying market features (city events, competitor price, pickup pace) are
  the main untapped lift; they slot straight onto the day axis.
- The promotion decision uses §5's signal-beats-noise gate; wire the winner into
  `src.scoring` / the app once the static notebooks (01-03) are back on the
  walk-forward regime.